# Figure 8: [Accessible] cumulative unique proof categories grid

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.plots.style import apply_paper_style, model_label, model_color, ordered_models
apply_paper_style()


In [ ]:
import matplotlib.pyplot as plt

from src.data_loader import load_proofs, try_load, warn_incomplete_coverage
from src.plots.proof_categories import cumulative_unique_categories
from src.plots.style import PROBLEM_IDS_ALL

proofs = try_load(load_proofs, label="proofs")
if proofs is not None:
    warn_incomplete_coverage(proofs, label="proofs", col="problem_id", expected=PROBLEM_IDS_ALL)
problem_ids = PROBLEM_IDS_ALL  # force the full 3x3 grid; missing problems render as blank "no data" panels


In [ ]:
if proofs is None:
    print("Skipping Fig 8: no proofs data loaded.")
else:
    fig, axes = plt.subplots(3, 3, figsize=(11, 9), sharex=True, sharey=True)
    empty_panels = []
    for ax, pid in zip(axes.flatten(), problem_ids):
        cum = cumulative_unique_categories(proofs, pid, max_samples=10)
        if cum.empty:
            empty_panels.append(pid)
            ax.text(0.5, 0.5, "no data", ha="center", va="center", fontsize=8, color="gray", transform=ax.transAxes)
        for m in ordered_models(cum["model_version"].unique()):
            msub = cum[cum["model_version"] == m].sort_values("x")
            ax.plot(msub["x"], msub["cum_unique"], color=model_color(m), label=model_label(m), linewidth=1.2)
        ax.set_title(f"Problem {pid}", fontsize=10)

    if empty_panels:
        print(f"No data for problem(s): {empty_panels}")

    handles, labels = [], []
    for ax in axes.flatten():
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            break
    if handles:
        fig.legend(handles, labels, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.05))
    else:
        print("No data at all: every panel is empty.")
    fig.tight_layout()
    plt.show()
